## Exemplo de Simulação Externa

Este notebook demonstra como usar a função `run_external_simulation` para testar uma estratégia de trading com dados e modelos definidos aqui, de forma independente do fluxo principal `run.py`.

In [ ]:
import pandas as pd
import yaml
import sys
import importlib
from pathlib import Path

# Adiciona a pasta 'src' ao path para permitir as importações dos nossos módulos
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    
from src.data_handler.provider import YFinanceProvider, MetaTraderProvider
from src.strategies.sentiment_lstm import SentimentLSTMStrategy
from src.backtest_engine.runner import run_external_simulation


### 1. Carregar Configurações e Dados

In [ ]:
# Carrega as configurações do projeto
import logging


with open('../configs/main.yaml', 'r') as file:
    config = yaml.safe_load(file)

# 2. Obter dados de mercado para os dois períodos
provider_name = config['data_settings'].get('provider', 'metatrader5') # Padrão para MetaTrader5
if provider_name == 'metatrader5':
    provider = MetaTraderProvider()
    logging.info("Usando o provedor de dados: MetaTrader 5")
else:
    provider = YFinanceProvider()
    logging.info("Usando o provedor de dados: Yahoo Finance")

train_cfg = config['data_settings']['in_sample']
market_data_is = provider.get_data(
    ticker=config['data_settings']['ticker'],
    start_date=train_cfg['start_date'],
    end_date=train_cfg['end_date'],
    sentiment_ticker=config['data_settings'].get('sentiment_ticker', '')
)

# 3. Carregar a estratégia dinamicamente
try:
    strategy_name = config['backtest_settings']['strategy_name']
    module_path = f"src.strategies.{config['backtest_settings']['strategy_module']}"
    strategy_module = importlib.import_module(module_path)
    StrategyClass = getattr(strategy_module, strategy_name)
    strategy_instance = StrategyClass()
except (ImportError, AttributeError) as e:
    logging.error(f"Não foi possível carregar a estratégia. Erro: {e}")
    raise

# Define os períodos de treino e teste
TRAIN_START = "2022-01-01"
TRAIN_END = "2023-12-31"
TEST_START = "2024-01-01"
TEST_END = "2024-12-31"

ticker = "SPY"

# Busca os dados de treino e teste
train_data = provider.get_data(ticker, TRAIN_START, TRAIN_END)
test_data = provider.get_data(ticker, TEST_START, TEST_END)

print(f"Dados de treino carregados: {len(train_data)} linhas")
print(f"Dados de teste carregados: {len(test_data)} linhas")

### 2. Treinar o Modelo

In [ ]:
# Instancia a estratégia
strategy = SentimentLSTMStrategy()

# Prepara os dados de treino
featured_data_train = strategy.define_features(train_data)
featured_data_train['target'] = (featured_data_train['close'].shift(-1) > featured_data_train['close']).astype(int)
featured_data_train = featured_data_train.dropna()

X_train = featured_data_train[strategy.get_feature_names()]
y_train = featured_data_train['target']

# Cria e treina o modelo
model = strategy.define_model()
print("Treinando o modelo...")
model.fit(X_train, y_train)
print("Modelo treinado com sucesso.")

### 3. Executar a Simulação Externa

In [ ]:
# Chama a nova função para executar a simulação nos dados de teste
trades_log = run_external_simulation(
    model=model,
    strategy=strategy,
    market_data=test_data,
    config=config
)

# Exibe o log de trades resultante
if not trades_log.empty:
    display(trades_log)
else:
    print("Nenhuma operação foi executada na simulação.")